# Modeling and Evaluation

This notebook consumes the window-level feature table produced by Person 2. The expected input is `data/processed/model_features.csv` with a `condition` column, numeric EEG features, and optional `subject_id`, `session_id`, and `window_id` metadata.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == '03_modeling':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.load_data import load_feature_data
from src.personal_baseline.build_baseline import build_personal_baseline
from src.models.general_model import train_general_models, save_general_model
from src.models.personalized_model import train_personalized_models, save_personalized_model
from src.evaluation.evaluate_models import (
    calculate_metrics,
    compare_models,
    save_comparison_report,
    create_confusion_matrix,
)

Evaluation module loaded successfully.


In [2]:
feature_path = PROJECT_ROOT / 'data' / 'processed' / 'model_features.csv'
data = load_feature_data(feature_path)
print(f'Loaded {len(data):,} windows and {len(data.columns):,} columns')
display(data.head())

Loaded 14,250 windows and 34 columns


,subject_id,session_id,window_id,condition,FT9_beta_relative_power,O1_mean_absolute,AFz_mean_absolute,PO7_mean_absolute,FC4_beta_relative_power,O1_rms,...,POz_std,PO7_rms,PO7_std,FC2_beta_relative_power,FT7_gamma_relative_power,AFz_beta_relative_power,FT8_beta_relative_power,FC6_gamma_relative_power,Fz_mean_absolute,FT10_gamma_relative_power
0,sub-10,ses-1,sub-10_ses-1_0,rest,0.182219,0.000007,0.000005,0.000006,0.324934,0.000009,...,0.000010,0.000008,0.000008,0.224906,0.063657,0.208203,0.254057,0.097555,0.000004,0.176413
1,sub-10,ses-1,sub-10_ses-1_1,rest,0.104684,0.000009,0.000007,0.000008,0.142336,0.000011,...,0.000011,0.000010,0.000010,0.128948,0.039517,0.112070,0.035395,0.039948,0.000006,0.052574
2,sub-10,ses-1,sub-10_ses-1_2,rest,0.087868,0.000009,0.000007,0.000009,0.110650,0.000012,...,0.000010,0.000012,0.000012,0.131793,0.085958,0.108508,0.034690,0.043688,0.000005,0.047688
3,sub-10,ses-1,sub-10_ses-1_3,rest,0.172088,0.000007,0.000005,0.000007,0.176935,0.000009,...,0.000008,0.000009,0.000009,0.186967,0.075244,0.150023,0.168488,0.101205,0.000003,0.036539
4,sub-10,ses-1,sub-10_ses-1_4,rest,0.264489,0.000007,0.000005,0.000006,0.312439,0.000009,...,0.000006,0.000008,0.000008,0.301416,0.052371,0.134298,0.154250,0.134671,0.000004,0.045580


## General population model

When `subject_id` is available, the trainer evaluates on held-out subjects to avoid leaking subject-specific patterns across the split.

In [3]:
(general_model, general_name, general_features, general_X_test, general_y_test, general_results) = train_general_models(data)
save_general_model(
    general_model,
    general_name,
    general_features,
    str(PROJECT_ROOT / 'models' / 'general_model.pkl')
)


Training: Logistic Regression
Accuracy  : 0.4681
Precision : 0.4955
Recall    : 0.4681
F1 Score  : 0.4406

Training: SVM
Accuracy  : 0.4814
Precision : 0.4894
Recall    : 0.4814
F1 Score  : 0.4646

Training: Random Forest
Accuracy  : 0.4354
Precision : 0.4366
Recall    : 0.4354
F1 Score  : 0.4200

BEST GENERAL MODEL: SVM

General model saved to:
C:\Users\Shanmukha\Desktop\eeg-cognitive-state-prediction\models\general_model.pkl


## Personalized model

The personalized model is trained for one subject using that subject's `rest` windows as the baseline. Set `subject_id` to the subject whose baseline should be used.

In [4]:
if 'subject_id' not in data.columns:
    raise ValueError('Person 2 output must include subject_id for personalized modeling.')

subject_id = data['subject_id'].iloc[0]
subject_data = data[data['subject_id'].astype(str) == str(subject_id)].copy()
baseline = build_personal_baseline(subject_data, subject_id)
(personalized_model, personalized_name, personalized_features, personalized_X_test, personalized_y_test, personalized_results) = train_personalized_models(subject_data, baseline)
save_personalized_model(
    personalized_model,
    personalized_name,
    personalized_features,
    baseline,
    str(PROJECT_ROOT / 'models' / 'personalized_model.pkl')
)


Training Personalized Logistic Regression
Accuracy  : 0.6632
Precision : 0.6682
Recall    : 0.6632
F1 Score  : 0.6646

Training Personalized SVM
Accuracy  : 0.7474
Precision : 0.7568
Recall    : 0.7474
F1 Score  : 0.7491

Training Personalized Random Forest
Accuracy  : 0.7105
Precision : 0.7302
Recall    : 0.7105
F1 Score  : 0.7128

BEST PERSONALIZED MODEL: SVM

Personalized model saved to:
C:\Users\Shanmukha\Desktop\eeg-cognitive-state-prediction\models\personalized_model.pkl


In [5]:
general_metrics = calculate_metrics(
    general_y_test,
    general_model.predict(general_X_test)
)
personalized_metrics = calculate_metrics(
    personalized_y_test,
    personalized_model.predict(personalized_X_test)
)
comparison = compare_models(general_metrics, personalized_metrics)
display(comparison)
save_comparison_report(
    comparison,
    str(PROJECT_ROOT / 'results' / 'reports' / 'model_comparison.csv')
)
create_confusion_matrix(
    general_y_test,
    general_model.predict(general_X_test),
    title='General Model Confusion Matrix',
    output_path=str(PROJECT_ROOT / 'results' / 'figures' / 'general_confusion_matrix.png')
)
create_confusion_matrix(
    personalized_y_test,
    personalized_model.predict(personalized_X_test),
    title='Personalized Model Confusion Matrix',
    output_path=str(PROJECT_ROOT / 'results' / 'figures' / 'personalized_confusion_matrix.png')
)

,General Model,Personalized Model,Improvement
Accuracy,0.481404,0.747368,0.265965
Precision,0.489421,0.756812,0.267391
Recall,0.481404,0.747368,0.265965
F1 Score,0.464572,0.749079,0.284507


Comparison report saved to: C:\Users\Shanmukha\Desktop\eeg-cognitive-state-prediction\results\reports\model_comparison.csv
Confusion matrix saved to: C:\Users\Shanmukha\Desktop\eeg-cognitive-state-prediction\results\figures\general_confusion_matrix.png
Confusion matrix saved to: C:\Users\Shanmukha\Desktop\eeg-cognitive-state-prediction\results\figures\personalized_confusion_matrix.png
